# 2. Lecke - A Microsoft Agent keretrendszer felfedezése

A **Microsoft Agent Framework (MAF)** egy egységes keretrendszer AI ügynökök építéséhez. Tiszta, komponálható architektúrát kínál négy alapvető építőelemmel:

- **Kliens** – kapcsolódik egy AI modell végponthoz és kezeli a kommunikációt
- **Ügynök** – burkolja a klienst utasításokkal és eszközdefiníciókkal
- **Eszközök** – bővítik az ügynök képességeit egyedi függvényekkel, amelyeket a modell hívhat meg
- **Munkamenet** – megőrzi a beszélgetés előzményeit többfordulós interakciókhoz

Ebben a leckében egy **utazási foglalási ügynököt** építünk, amely ellenőrzi az úticél elérhetőségét ezen fogalmak segítségével.


## Beállítás


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Az Agent Framework architektúrájának megértése

A Microsoft Agent Framework rétegzett architektúrát követ:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Ügyfél** – Egy `FoundryChatClient` csatlakozik egy Azure OpenAI telepítéshez. Kezeli az autentikációt, a kérések formázását és a válaszok elemzését.
2. **Agent** – Az ügyféltől a `provider.create_agent()` segítségével létrehozott agent, amely ötvözi a modell-hozzáférést az utasításokkal (rendszer prompt) és az eszközökkel.
3. **Eszközök** – `@tool` dekorátorral ellátott Python függvények, amelyeket az agent meghívhat műveletek végrehajtásához vagy adatok lekéréséhez.
4. **Munkamenet** – Egy `AgentSession` objektum (az `agent.create_session()` segítségével létrehozva), amely tárolja a beszélgetési előzményeket, lehetővé téve a többszörös fordulós párbeszédet, ahol az agent emlékszik a korábbi kontextusra.

Építsük fel lépésről lépésre az egyes rétegeket.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Eszközök hozzáadása az @tool dekorátorral

Az eszközök lehetővé teszik, hogy az ügynökök a szöveg generálásán túl más műveleteket is végezzenek. Az `@tool` dekorátor egy szokásos Python függvényt alakít át olyanná, amit az ügynök hívhat.

Főbb pontok:
- Használd az `Annotated[type, "description"]` formátumot, hogy a modell megértse az egyes paramétereket.
- A docstring lesz az eszköz leírása, amit a modell lát.
- Az `approval_mode="never_require"` azt jelenti, hogy az eszköz automatikusan fut, felhasználói jóváhagyás nélkül.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Egy ügynök létrehozása eszközökkel

Most összekapcsoljuk az ügyfelet, az utasításokat és az eszközöket egy ügynökbe. Az `instructions` szolgálnak rendszerkériésként — meghatározzák az ügynök személyiségét és viselkedését.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Többfordulós beszélgetések munkamenetekkel

Egy `AgentSession` (amelyet az `agent.create_session()` hoz létre) követi a beszélgetés összes üzenetét. Ha ugyanazt a munkamenetet adjuk át minden `agent.run()` hívásnak, az ügynök hozzáférhet a teljes beszélgetési előzményekhez, és visszautalhat korábbi üzenetekre.

A `tools=[check_destination_availability]` paramétert adjuk át, hogy az ügynök minden fordulóban meghívhassa az elérhetőség-ellenőrzőnket.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Összefoglaló

Ebben a leckében megismerkedtél a Microsoft Agent Framework négy pillérével:

| Fogalom | Amit megtanultál |
|---------|------------------|
| **Ügyfél** | A `FoundryChatClient` hitelesítés alapú kapcsolódást biztosít az Azure OpenAI-hoz |
| **Ügynök** | A `provider.create_agent()` egy modellt, utasításokat és nevet csomagol össze |
| **Eszközök** | Az `@tool` dekorátor Python függvényeket tesz elérhetővé, amelyeket az ügynök hívhat |
| **Munkamenet** | Az `agent.create_session()` több körös beszélgetés történetét kezeli |

Ezek az építőelemek együtt alkotnak olyan ügynököket, amelyek természetes beszélgetést folytathatnak, külső függvényeket hívhatnak, és megőrizhetik a kontextust — ez az alapja a későbbi leckék fejlettebb ügynök mintázatainak.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
